# s10 — MCP toolsets

**What this teaches:** how external Model Context Protocol (MCP) servers are mounted onto a Omnis agent as plain tools. A single JSON file (`config/mcp_config.json`) declares each server (command + args + env); the loader spawns them, negotiates the protocol, and exposes their tools alongside the built-in ones. The model can't tell the difference.

**Concept anchor:** an MCP server is just a remote toolbox. Once mounted, its tools live in the same loop as `read`, `write`, and `bash` — see [s05_tools](../s05_tools/s05_tools.ipynb) for the local equivalent.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [s01_loop](../s01_loop/s01_loop.ipynb)).
- `config/mcp_config.json` must exist at the path passed to `mcpcfg.Load`. The default points at the repo-root file; if you launch Jupyter from a different cwd you may need to adjust the path or copy the file.
- Whatever commands the YAML invokes must be installed and reachable on `PATH`.

## 1. Imports

`mcpcfg` is the internal loader: it parses the YAML, spawns each server, and hands back ADK-compatible toolsets ready to mount.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    mcpcfg "github.com/blouargant/omnis/internal/mcp"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Load the MCP config

`mcpcfg.Load` parses the YAML into a `Config` struct — names, commands, args, env. Nothing has been started yet; this is just deserialisation. Treat any parse error as a stop sign.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
cfg, err := mcpcfg.Load("config/mcp_config.json")
must(err)
fmt.Println("mcp config loaded")

## 4. Spawn the servers and build toolsets

`cfg.Toolsets()` does the heavy lifting: for each server declared in the YAML it spawns the process, performs the MCP handshake, enumerates the server's tools, and wraps the whole bundle as an ADK *toolset*. A toolset behaves like a group of tools: the model sees them by name; the runner routes calls through MCP.

In [ ]:
tsets, err := cfg.Toolsets()
must(err)
fmt.Println("toolsets ready:", len(tsets))

## 5. Mount on the agent

Toolsets are passed via the `Toolsets` field (as opposed to `Tools`, which is for individually-built tools). Otherwise the agent and runner are identical to every other example — the loop is unchanged.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s10_mcp",
    Description: "MCP-tool-aware agent.",
    Instruction: "List the MCP tools you have, grouped by server.",
    Model:       llm,
    Toolsets:    tsets,
})
must(err)
r, err := agentkit.Runner("s10", a)
must(err)
fmt.Println("agent + runner ready")

## 6. Ask the agent what it has

The model can introspect its own tool catalogue, so a simple prompt produces a grouped listing. Watch for tool names prefixed by their MCP server — that's the convention the loader uses to keep namespaces clean.

In [ ]:
prompt := "What MCP tools do you have?"
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The reply groups tools by server. Each one was discovered live at startup — the YAML did not list them.
- Stop the kernel and the MCP child processes die with it; `cfg.Toolsets()` ties their lifecycle to the parent. For a deeper view of how tools dispatch, cross-link to [s05_tools](../s05_tools/s05_tools.ipynb).

## Try it yourself

1. Add a new server entry in `config/mcp_config.json` (e.g. an in-tree stub) and rerun sections 3–6 — the new tools show up without any Go change.
2. Change the prompt to actually *call* one of the discovered tools (e.g. "use the X tool to fetch Y") and trace the loop as it dispatches through MCP.